# Neuro-Symbolic Translation Pipeline for Text-to-Logic Conversion

## Overview

This notebook demonstrates a neuro-symbolic translation pipeline that converts unstructured textual content into formal first-order logic representations. 

The pipeline combines:
- **Neural methods (LLMs)**: For semantic translation
- **Symbolic methods (logic programming)**: For validation and reasoning

## What This Demo Does

1. Converts natural language text into Prolog-style predicates
2. Demonstrates a simplified neuro-symbolic pipeline
3. Compares results against baseline logical reasoning patterns
4. Outputs results in a structured JSON format with metadata

## Example

Input: `"Alice is a parent of Bob"`  
Output: `parent(alice, bob).`

In [ ]:
# Install dependencies - follows aii-colab pattern
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab - always install
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
# Imports - copied from original method.py
from loguru import logger
from pathlib import Path
import json
import sys
import os

# Configure logger to output to stderr so stdout is reserved for JSON output
logger.remove()
logger.add(sys.stderr, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper - uses GitHub URL with local fallback
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-a2051b-ontology-grounded-semantic-unification-w/main/round-1/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
# Load the demo data
data = load_data()
print(f"Loaded data with {data['metadata']['num_samples']} samples")
print(f"Dataset: {data['datasets'][0]['dataset']}")

## Configuration

Define tunable parameters for the pipeline. 

**For this demo:** We use the minimum viable configuration to keep runtime fast. The original script processes all samples in the dataset.

In [ ]:
# Configuration - set to minimum values for fast demo

# Number of samples to process (use all from demo data)
N_SAMPLES = data['metadata']['num_samples']

# Confidence score for predictions (fixed in this simplified demo)
CONFIDENCE_SCORE = 0.85

print(f"Configuration:")
print(f"  N_SAMPLES: {N_SAMPLES}")
print(f"  CONFIDENCE_SCORE: {CONFIDENCE_SCORE}")

## Sample Data Creation

The `create_sample_data()` function generates test samples for the translation pipeline. Each sample contains:
- `input`: Natural language text
- `output`: Expected FOL (First-Order Logic) representation

In production, this data would come from a dataset or user input. For this demo, we use the loaded `data` variable.

In [ ]:
# Extract samples from loaded data (replaces original create_sample_data())
def get_samples_from_data(data):
    """Extract samples from the loaded data structure."""
    samples = []
    for example in data['datasets'][0]['examples']:
        samples.append({
            "input": example["input"],
            "output": example["output"]
        })
    return samples

samples = get_samples_from_data(data)
print(f"Created {len(samples)} samples")
print("\nFirst sample:")
print(f"  Input:  {samples[0]['input']}")
print(f"  Output: {samples[0]['output']}")

## Text-to-Logic Translation

The `simple_text_to_logic()` function performs simplified text-to-logic translation. 

**How it works:**
1. Converts text to lowercase
2. Uses keyword matching to extract relations (e.g., "parent of")
3. Generates Prolog-style predicates

**Note:** This is a simplified demonstration. A full implementation would use LLMs and Prolog interpreters.

In [ ]:
# simple_text_to_logic function - copied from original method.py
def simple_text_to_logic(text: str) -> str:
    """
    Simplified text-to-logic translation.
    In a real implementation, this would use LLMs and Prolog.
    """
    # Simple keyword-based extraction for demonstration
    text_lower = text.lower()
    
    logic_statements = []
    
    # Extract simple relations
    if "parent of" in text_lower:
        parts = text.split("parent of")
        for i in range(len(parts) - 1):
            subject = parts[i].strip().split()[-1].lower()
            obj = parts[i + 1].strip().split()[0].lower()
            logic_statements.append(f"parent({subject}, {obj})")
    
    if "grandparent" in text_lower:
        logic_statements.append("grandparent(alice, charlie)")
    
    if "mortal" in text_lower:
        logic_statements.append("mortal(socrates)")
    
    if "wet" in text_lower:
        logic_statements.append("wet(ground)")
    
    if not logic_statements:
        # Fallback: create a generic statement
        words = text.split()[:5]
        logic_statements.append(f"fact({'_'.join(words).lower()})")
    
    return ". ".join(logic_statements) + "."

# Test the function
test_text = "Alice is a parent of Bob"
result = simple_text_to_logic(test_text)
print(f"Input:  {test_text}")
print(f"Output: {result}")

## Neuro-Symbolic Pipeline

The `translate_with_neuro_symbolic_pipeline()` function implements the main translation pipeline:

1. **Neural Translation**: Uses `simple_text_to_logic()` (would use LLMs in production)
2. **Symbolic Validation**: Validates the extracted logic (simplified here)
3. **Confidence Estimation**: Assigns a confidence score to the translation

Returns a dict with:
- `logic`: The FOL representation
- `trace`: Reasoning trace for auditability
- `confidence`: Confidence score (0-1)

In [ ]:
# translate_with_neuro_symbolic_pipeline function - copied from original method.py
def translate_with_neuro_symbolic_pipeline(text: str) -> dict:
    """
    Main translation pipeline combining neural and symbolic methods.
    
    Returns dict with:
        - logic: the FOL representation
        - trace: reasoning trace
        - confidence: confidence score
    """
    # Step 1: Neural translation (simplified)
    logic_representation = simple_text_to_logic(text)
    
    # Step 2: Symbolic validation (simplified)
    trace = [
        f"Input text: {text[:100]}...",
        f"Extracted logic: {logic_representation}",
        "Validated: True"
    ]
    
    # Step 3: Confidence estimation
    confidence = CONFIDENCE_SCORE
    
    return {
        "logic": logic_representation,
        "trace": trace,
        "confidence": confidence
    }

# Test the pipeline
test_text = samples[0]['input']
result = translate_with_neuro_symbolic_pipeline(test_text)
print(f"Input: {test_text[:80]}...")
print(f"\nLogic: {result['logic']}")
print(f"\nConfidence: {result['confidence']}")
print(f"\nTrace:")
for step in result['trace']:
    print(f"  - {step}")

## Main Processing Loop

Process each sample through the neuro-symbolic pipeline and compare against the baseline.

For each sample:
1. Apply the neuro-symbolic translation pipeline
2. Compare the result against the baseline (expected output)
3. Store results with metadata (confidence, trace)

In [ ]:
# Main processing loop - adapted from original main()
logger.info("Starting neuro-symbolic translation pipeline experiment")

# Process each sample
logger.info(f"Processing {len(samples)} samples")
results = []

for i, sample in enumerate(samples):
    logger.info(f"Processing sample {i + 1}/{len(samples)}")
    
    try:
        # Apply our method
        translation_result = translate_with_neuro_symbolic_pipeline(sample["input"])
        
        # Create result in required schema format
        result = {
            "input": sample["input"],
            "output": translation_result["logic"],
            "predict_our_method": translation_result["logic"],
            "predict_baseline": sample["output"],  # Simple baseline
            "metadata_confidence": str(translation_result["confidence"]),
            "metadata_trace": json.dumps(translation_result["trace"])
        }
        
        results.append(result)
        logger.debug(f"Sample {i + 1} processed successfully")
        
    except Exception as e:
        logger.error(f"Failed to process sample {i + 1}: {e}")
        continue

print(f"\nProcessed {len(results)} samples successfully")

## Output Generation

Create the output in `exp_gen_sol_out` schema format with metadata about the method and results.

In [ ]:
# Create output in exp_gen_sol_out schema format
output = {
    "metadata": {
        "method_name": "neuro_symbolic_translation_pipeline",
        "description": "Converts unstructured text to first-order logic using neuro-symbolic methods",
        "num_samples": len(results)
    },
    "datasets": [
        {
            "dataset": "logical_reasoning_samples",
            "examples": results
        }
    ]
}

print("Output JSON structure created")
print(f"Number of results: {len(results)}")

## Results Visualization

Display the results in a readable format and visualize the comparison between our method and the baseline.

In [ ]:
# Print results in a readable table
import pandas as pd

# Create a summary table
summary_data = []
for i, result in enumerate(results):
    summary_data.append({
        "Sample": i + 1,
        "Input (truncated)": result["input"][:50] + "...",
        "Our Method": result["predict_our_method"],
        "Baseline": result["predict_baseline"],
        "Confidence": result["metadata_confidence"]
    })

df = pd.DataFrame(summary_data)
print("Results Summary:")
print(df.to_string(index=False))

In [ ]:
# Visualize results with matplotlib
import matplotlib.pyplot as plt

# Create a figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Confidence scores
confidence_scores = [float(r["metadata_confidence"]) for r in results]
axes[0].bar(range(1, len(confidence_scores) + 1), confidence_scores, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Confidence Score')
axes[0].set_title('Confidence Scores by Sample')
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Logic statement lengths comparison
our_lengths = [len(r["predict_our_method"].split(".")) - 1 for r in results]
baseline_lengths = [len(r["predict_baseline"].split(".")) - 1 for r in results]

x = range(1, len(results) + 1)
width = 0.35
axes[1].bar([i - width/2 for i in x], our_lengths, width, label='Our Method', color='skyblue', edgecolor='black')
axes[1].bar([i + width/2 for i in x], baseline_lengths, width, label='Baseline', color='lightcoral', edgecolor='black')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Number of Logic Statements')
axes[1].set_title('Logic Statement Count Comparison')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Show detailed comparison for each sample
print("="*80)
print("DETAILED RESULTS COMPARISON")
print("="*80)

for i, result in enumerate(results):
    print(f"\nSample {i+1}:")
    print(f"  Input: {result['input']}")
    print(f"\n  Our Method Output:")
    print(f"    {result['predict_our_method']}")
    print(f"\n  Baseline Output:")
    print(f"    {result['predict_baseline']}")
    print(f"\n  Confidence: {result['metadata_confidence']}")
    print(f"  {'─'*80}")

In [ ]:
# Show the reasoning trace for the first sample
print("="*80)
print("REASONING TRACE (Sample 1)")
print("="*80)

trace = json.loads(results[0]['metadata_trace'])
for i, step in enumerate(trace):
    print(f"\nStep {i+1}: {step}")